In [0]:
%run ../Includes/Copy-Datasets

In [0]:
(spark.read.table("bronze").filter("topic = 'orders'").count())

##### For `Static Data` we can simply use `'dropDuplicates()'` function to eliminate duplicate recprds

In [0]:
from pyspark.sql import functions as F

json_schema  = "order_id STRING, order_timestamp TIMESTAMP, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, sub_total BIGINT>>"


batch_total = (spark.read
                .table("bronze")
                .filter("topic = 'orders'")
                .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
                .select("v.*")
                .dropDuplicates(["order_id","order_timestamp"])
                .count()
)
 
print(batch_total)

In Spark Structured Streaming : 
- We can also use dropDuplicates() function.
- Structured Streaming can track state information for the unique keys in the data.
- This ensures that duplicate records do not exist within or between microbatches.
- However, over time, this state information will scale to represent all history.
- We can limit the amount of state to be maintained by using watermarking.
- Watemarking allows to only track state information for a window of time in which we expect records could be delayed.
- In this way, we are sure that there is no duplicate records exist in each new microbatches to be processed.

In [0]:
deduped_df = (spark.readStream
                .table("bronze")
                .filter("topic = 'orders'")
                .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
                .select("v.*")
                .withWatermark("order_timestamp", "30 seconds")
                .dropDuplicates(["order_id","order_timestamp"])
)

- For de-duplication in streaming we need to ensure inserted records are not already in the target table.
- We can achieve this using **insert-only merge**.

In [0]:
def upsert_data(microBatchDF, batch):
    microBatchDF.createOrReplaceTempView("orders_microbatch")

    sql_query = """
                    MERGE INTO orders_silver a
                    USING orders_microbatch b
                    ON a.order_id = b.order_id AND a.order_timestamp = b.order_timestamp
                    WHEN NOT MATCHED THEN INSERT *
    """

    microBatchDF.sparkSession.sql(sql_query)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS orders_silver
(order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>)

In [0]:
query = (deduped_df.writeStream
        .foreachBatch(upsert_data)
        .option("checkpointLocation",f"{bookstore.checkpoint_path}/orders_silver")
        .trigger(availableNow=True)
        .start()
                
)

query.awaitTermination()

In [0]:
streaming_total = spark.read.table("orders_silver").count()

print(f"batch total: {batch_total}")
print(f"streaming total: {streaming_total}")
